A. Studi Kasus: Optimasi Penentuan Titik Hub Logistik & Analisis Jangkauan Spasial (Buffer Coverage)
Latar Belakang & Masalah: Perusahaan ekspedisi ingin memperluas jaringan pengantaran paket di suatu wilayah perkotaan (misal area Bengkulu/Sumatera). Mereka memiliki ratusan titik sebaran pemesan. Jika hub ditempatkan secara sembarangan, biaya armada membengkak dan waktu antar melebihi batas toleransi.
Solusi:
Menggunakan rumus trigonometri Haversine untuk menghitung jarak nyata melengkung di permukaan bola bumi (
𝑅
=
6.371
 km
R=6.371 km).
Menerapkan pengelompokan spasial (Spatial K-Means Clustering) untuk menemukan koordinat optimal (centroid/medoid) bagi hub baru.
Mengukur persentase titik pemesan yang masuk ke dalam zona penyangga (buffer zone) berjarak
≤
10
 km
≤10 km dari hub terdekat.

In [1]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans

# Formula Jarak Haversine (Jarak permukaan bumi sferis antar dua koordinat GPS)
def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0 # Jari-jari bumi dalam kilometer
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)

    a = np.sin(dphi / 2.0)**2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda / 2.0)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return R * c

def studi_kasus_3_industrial_gis():
    np.random.seed(42)
    n_titik = 100

    # Koordinat sebaran titik permintaan di sekitar Bengkulu (-3.7928, 102.2608)
    base_lat, base_lon = -3.7928, 102.2608
    lats = base_lat + np.random.normal(0, 0.07, n_titik)
    lons = base_lon + np.random.normal(0, 0.07, n_titik)
    demands = np.random.randint(15, 120, n_titik) # Jumlah paket per titik

    df_spasial = pd.DataFrame({'Lat': lats, 'Lon': lons, 'Demand': demands})

    # 1. Optimasi Spasial: Penempatan 2 Distribution Hub
    kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
    df_spasial['Hub_Assigned'] = kmeans.fit_predict(df_spasial[['Lat', 'Lon']])
    hub_coords = kmeans.cluster_centers_

    # 2. Analisis Buffer Coverage (Radius Maksimal Armada = 10 km)
    buffer_radius_km = 10.0
    covered = 0
    total_travel = 0.0

    for _, row in df_spasial.iterrows():
        hub_idx = int(row['Hub_Assigned'])
        h_lat, h_lon = hub_coords[hub_idx]
        d = haversine(row['Lat'], row['Lon'], h_lat, h_lon)
        total_travel += d
        if d <= buffer_radius_km:
            covered += 1

    print("=== ANALISIS OPTIMASI SPASIAL & GIS LOGISTIK ===")
    print(f"Total Titik Konsumen: {n_titik}")
    print("Koordinat Optimal Hub yang Direkomendasikan:")
    for i, (h_lat, h_lon) in enumerate(hub_coords, 1):
        print(f"  - Hub {i}: Latitude {h_lat:.4f}, Longitude {h_lon:.4f}")
    print(f"\nCakupan Layanan Buffer (<= {buffer_radius_km} km): {covered}/{n_titik} ({covered/n_titik*100:.1f}%)")
    print(f"Rata-rata Jarak Tempuh Pengiriman: {total_travel/n_titik:.2f} km")

if __name__ == '__main__':
    studi_kasus_3_industrial_gis()

=== ANALISIS OPTIMASI SPASIAL & GIS LOGISTIK ===
Total Titik Konsumen: 100
Koordinat Optimal Hub yang Direkomendasikan:
  - Hub 1: Latitude -3.8438, Longitude 102.3146
  - Hub 2: Latitude -3.7709, Longitude 102.2275

Cakupan Layanan Buffer (<= 10.0 km): 78/100 (78.0%)
Rata-rata Jarak Tempuh Pengiriman: 7.09 km
